In [54]:
from pathlib import Path
import pandas as pd

print("Python 环境和 pandas 已成功启动。")
print("pandas version:", pd.__version__)

Python 环境和 pandas 已成功启动。
pandas version: 3.0.3


In [55]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()

# Notebook 有时从项目根目录运行，有时从 notebooks/ 文件夹运行。
# 这段代码会自动找到项目根目录。
if not (PROJECT_ROOT / "data" / "raw").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

csv_files = sorted(RAW_DATA_DIR.glob("*.csv"))

print("项目根目录：")
print(PROJECT_ROOT)

print("\n原始数据目录：")
print(RAW_DATA_DIR)

print("\n找到的 CSV 文件数量：", len(csv_files))

for file in csv_files:
    print("-", file.name)

项目根目录：
/Users/sarah/Documents/data-analysis-projects/Olist公开电商数据的个人数据分析项目

原始数据目录：
/Users/sarah/Documents/data-analysis-projects/Olist公开电商数据的个人数据分析项目/data/raw

找到的 CSV 文件数量： 9
- olist_customers_dataset.csv
- olist_geolocation_dataset.csv
- olist_order_items_dataset.csv
- olist_order_payments_dataset.csv
- olist_order_reviews_dataset.csv
- olist_orders_dataset.csv
- olist_products_dataset.csv
- olist_sellers_dataset.csv
- product_category_name_translation.csv


# Phase 1 — Data Understanding

## Purpose
This notebook checks the raw Olist CSV files before any cleaning, SQL analysis, or business analysis.

## Rules
- Do not delete rows.
- Do not fill missing values.
- Do not modify the raw CSV files.
- Do not make business conclusions yet.

## Checks included
1. Confirm the raw-data folder and CSV files.
2. Inspect table sizes, columns, samples, and data types.
3. Check key fields for missing values and duplicates.
4. Check missing values.
5. Check date ranges and categorical values.
6. Validate important table relationships.

In [56]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

print("Python executable:")
print(sys.executable)

print("\npandas version:")
print(pd.__version__)

Python executable:
/Users/sarah/Documents/data-analysis-projects/Olist公开电商数据的个人数据分析项目/.venv/bin/python

pandas version:
3.0.3


In [57]:
CURRENT_WORKING_DIRECTORY = Path.cwd().resolve()

possible_project_roots = [
    CURRENT_WORKING_DIRECTORY,
    CURRENT_WORKING_DIRECTORY.parent
]

PROJECT_ROOT = None
RAW_DATA_DIR = None

for possible_root in possible_project_roots:
    possible_raw_data_dir = possible_root / "data" / "raw"

    if possible_raw_data_dir.exists():
        PROJECT_ROOT = possible_root
        RAW_DATA_DIR = possible_raw_data_dir
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find data/raw/. "
        "Please confirm that your notebook is inside the project folder "
        "and that the raw CSV files are stored in data/raw/."
    )

print("Current working directory:")
print(CURRENT_WORKING_DIRECTORY)

print("\nProject root detected:")
print(PROJECT_ROOT)

print("\nRaw data folder detected:")
print(RAW_DATA_DIR)

Current working directory:
/Users/sarah/Documents/data-analysis-projects/Olist公开电商数据的个人数据分析项目/notebooks

Project root detected:
/Users/sarah/Documents/data-analysis-projects/Olist公开电商数据的个人数据分析项目

Raw data folder detected:
/Users/sarah/Documents/data-analysis-projects/Olist公开电商数据的个人数据分析项目/data/raw


In [58]:
EXPECTED_FILES = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

found_csv_paths = sorted(RAW_DATA_DIR.glob("*.csv"))
found_csv_names = {path.name for path in found_csv_paths}

missing_file_names = [
    file_name
    for file_name in EXPECTED_FILES.values()
    if file_name not in found_csv_names
]

unexpected_file_names = [
    path.name
    for path in found_csv_paths
    if path.name not in EXPECTED_FILES.values()
]

print(f"Found {len(found_csv_paths)} CSV files.\n")

print("CSV files found:")
for path in found_csv_paths:
    file_size_mb = path.stat().st_size / (1024 * 1024)
    print(f"- {path.name} ({file_size_mb:.2f} MB)")

print("\nMissing expected files:")
if missing_file_names:
    for file_name in missing_file_names:
        print(f"- {file_name}")
else:
    print("None")

print("\nUnexpected CSV files:")
if unexpected_file_names:
    for file_name in unexpected_file_names:
        print(f"- {file_name}")
else:
    print("None")

Found 9 CSV files.

CSV files found:
- olist_customers_dataset.csv (8.62 MB)
- olist_geolocation_dataset.csv (58.44 MB)
- olist_order_items_dataset.csv (14.72 MB)
- olist_order_payments_dataset.csv (5.51 MB)
- olist_order_reviews_dataset.csv (13.78 MB)
- olist_orders_dataset.csv (16.84 MB)
- olist_products_dataset.csv (2.27 MB)
- olist_sellers_dataset.csv (0.17 MB)
- product_category_name_translation.csv (0.00 MB)

Missing expected files:
None

Unexpected CSV files:
None


In [59]:
dataframes = {}

for table_name, file_name in EXPECTED_FILES.items():
    file_path = RAW_DATA_DIR / file_name

    dataframes[table_name] = pd.read_csv(
        file_path,
        low_memory=False
    )

    print(f"Loaded {table_name}: {dataframes[table_name].shape[0]:,} rows")
    

Loaded customers: 99,441 rows
Loaded geolocation: 1,000,163 rows
Loaded order_items: 112,650 rows
Loaded order_payments: 103,886 rows
Loaded order_reviews: 99,224 rows
Loaded orders: 99,441 rows
Loaded products: 32,951 rows
Loaded sellers: 3,095 rows
Loaded category_translation: 71 rows


In [60]:
customers = dataframes["customers"]
geolocation = dataframes["geolocation"]
order_items = dataframes["order_items"]
order_payments = dataframes["order_payments"]
order_reviews = dataframes["order_reviews"]
orders = dataframes["orders"]
products = dataframes["products"]
sellers = dataframes["sellers"]
category_translation = dataframes["category_translation"]

print("All table variables are ready.")

All table variables are ready.


In [61]:
table_overview_rows = []

for table_name, dataframe in dataframes.items():
    table_overview_rows.append(
        {
            "table_name": table_name,
            "rows": dataframe.shape[0],
            "columns": dataframe.shape[1],
            "column_names": ", ".join(dataframe.columns)
        }
    )

table_overview = pd.DataFrame(table_overview_rows)

display(table_overview)

,table_name,rows,columns,column_names
0,customers,99441,5,"customer_id, customer_unique_id, customer_zip_..."
1,geolocation,1000163,5,"geolocation_zip_code_prefix, geolocation_lat, ..."
2,order_items,112650,7,"order_id, order_item_id, product_id, seller_id..."
3,order_payments,103886,5,"order_id, payment_sequential, payment_type, pa..."
4,order_reviews,99224,7,"review_id, order_id, review_score, review_comm..."
5,orders,99441,8,"order_id, customer_id, order_status, order_pur..."
6,products,32951,9,"product_id, product_category_name, product_nam..."
7,sellers,3095,4,"seller_id, seller_zip_code_prefix, seller_city..."
8,category_translation,71,2,"product_category_name, product_category_name_e..."


In [62]:
for table_name, dataframe in dataframes.items():
    print("=" * 100)
    print(f"TABLE: {table_name}")
    print(f"SHAPE: {dataframe.shape[0]:,} rows × {dataframe.shape[1]} columns")

    print("\nColumn names and pandas data types:")
    data_type_table = (
        dataframe.dtypes
        .astype(str)
        .rename("data_type")
        .rename_axis("column_name")
        .reset_index()
    )
    display(data_type_table)

    print("\nFirst 3 rows:")
    display(dataframe.head(3))

TABLE: customers
SHAPE: 99,441 rows × 5 columns

Column names and pandas data types:


,column_name,data_type
0,customer_id,str
1,customer_unique_id,str
2,customer_zip_code_prefix,int64
3,customer_city,str
4,customer_state,str



First 3 rows:


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP


TABLE: geolocation
SHAPE: 1,000,163 rows × 5 columns

Column names and pandas data types:


...

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ


TABLE: category_translation
SHAPE: 71 rows × 2 columns

Column names and pandas data types:


,column_name,data_type
0,product_category_name,str
1,product_category_name_english,str



First 3 rows:


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


In [63]:
KEY_CHECKS = [
    {
        "check_name": "orders.order_id",
        "table_name": "orders",
        "columns": ["order_id"],
        "expected_rule": "Should be unique: one row should represent one order."
    },
    {
        "check_name": "customers.customer_id",
        "table_name": "customers",
        "columns": ["customer_id"],
        "expected_rule": "Should be unique: used to connect customers to orders."
    },
    {
        "check_name": "order_items.(order_id + order_item_id)",
        "table_name": "order_items",
        "columns": ["order_id", "order_item_id"],
        "expected_rule": "Should be unique together: one order-item detail row."
    },
    {
        "check_name": "order_items.order_id",
        "table_name": "order_items",
        "columns": ["order_id"],
        "expected_rule": "Duplicates are expected: one order can contain multiple items."
    },
    {
        "check_name": "order_payments.(order_id + payment_sequential)",
        "table_name": "order_payments",
        "columns": ["order_id", "payment_sequential"],
        "expected_rule": "Should be unique together: one payment sequence within one order."
    },
    {
        "check_name": "order_payments.order_id",
        "table_name": "order_payments",
        "columns": ["order_id"],
        "expected_rule": "Duplicates can be expected: one order may have multiple payment records."
    },
    {
        "check_name": "products.product_id",
        "table_name": "products",
        "columns": ["product_id"],
        "expected_rule": "Should be unique: one product record."
    },
    {
        "check_name": "sellers.seller_id",
        "table_name": "sellers",
        "columns": ["seller_id"],
        "expected_rule": "Should be unique: one seller record."
    },
    {
        "check_name": "order_reviews.review_id",
        "table_name": "order_reviews",
        "columns": ["review_id"],
        "expected_rule": "Expected to identify review records; inspect duplicates before deciding."
    },
    {
        "check_name": "order_reviews.order_id",
        "table_name": "order_reviews",
        "columns": ["order_id"],
        "expected_rule": "Relationship field; inspect duplicates and do not assume uniqueness yet."
    },
    {
        "check_name": "category_translation.product_category_name",
        "table_name": "category_translation",
        "columns": ["product_category_name"],
        "expected_rule": "Should be unique: one translation mapping per Portuguese category name."
    },
    {
        "check_name": "geolocation.geolocation_zip_code_prefix",
        "table_name": "geolocation",
        "columns": ["geolocation_zip_code_prefix"],
        "expected_rule": "Duplicates are expected: this field is not a strict unique primary key."
    }
]

key_check_rows = []

for check in KEY_CHECKS:
    dataframe = dataframes[check["table_name"]]
    key_columns = check["columns"]

    rows_with_missing_key_values = int(
        dataframe[key_columns].isna().any(axis=1).sum()
    )

    duplicate_rows_based_on_key = int(
        dataframe.duplicated(subset=key_columns).sum()
    )

    key_check_rows.append(
        {
            "check_name": check["check_name"],
            "table_name": check["table_name"],
            "key_columns": " + ".join(key_columns),
            "rows_with_missing_key_values": rows_with_missing_key_values,
            "duplicate_rows_based_on_key": duplicate_rows_based_on_key,
            "expected_rule": check["expected_rule"]
        }
    )

key_check_results = pd.DataFrame(key_check_rows)

display(key_check_results)

,check_name,table_name,key_columns,rows_with_missing_key_values,duplicate_rows_based_on_key,expected_rule
0,orders.order_id,orders,order_id,0,0,Should be unique: one row should represent one...
1,customers.customer_id,customers,customer_id,0,0,Should be unique: used to connect customers to...
2,order_items.(order_id + order_item_id),order_items,order_id + order_item_id,0,0,Should be unique together: one order-item deta...
3,order_items.order_id,order_items,order_id,0,13984,Duplicates are expected: one order can contain...
4,order_payments.(order_id + payment_sequential),order_payments,order_id + payment_sequential,0,0,Should be unique together: one payment sequenc...
5,order_payments.order_id,order_payments,order_id,0,4446,Duplicates can be expected: one order may have...
6,products.product_id,products,product_id,0,0,Should be unique: one product record.
7,sellers.seller_id,sellers,seller_id,0,0,Should be unique: one seller record.
8,order_reviews.review_id,order_reviews,review_id,0,814,Expected to identify review records; inspect d...
9,order_reviews.order_id,order_reviews,order_id,0,551,Relationship field; inspect duplicates and do ...


In [64]:
missing_value_rows = []

for table_name, dataframe in dataframes.items():
    for column_name in dataframe.columns:
        missing_count = int(dataframe[column_name].isna().sum())
        missing_percentage = round(
            missing_count / len(dataframe) * 100,
            2
        )

        missing_value_rows.append(
            {
                "table_name": table_name,
                "column_name": column_name,
                "missing_count": missing_count,
                "missing_percentage": missing_percentage
            }
        )

missing_value_results = pd.DataFrame(missing_value_rows)

missing_value_results_with_missing = (
    missing_value_results[
        missing_value_results["missing_count"] > 0
    ]
    .sort_values(
        by=["missing_count", "missing_percentage"],
        ascending=False
    )
)

if missing_value_results_with_missing.empty:
    print("No missing values were found in any table.")
else:
    display(missing_value_results_with_missing)

,table_name,column_name,missing_count,missing_percentage
25,order_reviews,review_comment_title,87656,88.34
26,order_reviews,review_comment_message,58247,58.70
35,orders,order_delivered_customer_date,2965,2.98
34,orders,order_delivered_carrier_date,1783,1.79
38,products,product_category_name,610,1.85
39,products,product_name_lenght,610,1.85
40,products,product_description_lenght,610,1.85
41,products,product_photos_qty,610,1.85
33,orders,order_approved_at,160,0.16
42,products,product_weight_g,2,0.01


In [65]:
FOCUS_FIELDS = {
    "orders": [
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "customers": [
        "customer_id",
        "customer_unique_id",
        "customer_city",
        "customer_state",
        "customer_zip_code_prefix"
    ],
    "order_items": [
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id",
        "price",
        "freight_value"
    ],
    "order_payments": [
        "order_id",
        "payment_sequential",
        "payment_type",
        "payment_value"
    ],
    "order_reviews": [
        "review_id",
        "order_id",
        "review_score",
        "review_creation_date"
    ],
    "products": [
        "product_id",
        "product_category_name",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ],
    "sellers": [
        "seller_id",
        "seller_city",
        "seller_state",
        "seller_zip_code_prefix"
    ],
    "category_translation": [
        "product_category_name",
        "product_category_name_english"
    ]
}

focus_missing_rows = []

for table_name, field_names in FOCUS_FIELDS.items():
    dataframe = dataframes[table_name]

    for field_name in field_names:
        missing_count = int(dataframe[field_name].isna().sum())
        missing_percentage = round(
            missing_count / len(dataframe) * 100,
            2
        )

        focus_missing_rows.append(
            {
                "table_name": table_name,
                "field_name": field_name,
                "missing_count": missing_count,
                "missing_percentage": missing_percentage
            }
        )

focus_missing_results = pd.DataFrame(focus_missing_rows)

display(
    focus_missing_results.sort_values(
        by=["table_name", "missing_count"],
        ascending=[True, False]
    )
)

,table_name,field_name,missing_count,missing_percentage
35,category_translation,product_category_name,0,0.00
36,category_translation,product_category_name_english,0,0.00
6,customers,customer_id,0,0.00
7,customers,customer_unique_id,0,0.00
8,customers,customer_city,0,0.00
9,customers,customer_state,0,0.00
10,customers,customer_zip_code_prefix,0,0.00
11,order_items,order_id,0,0.00
12,order_items,order_item_id,0,0.00
13,order_items,product_id,0,0.00


In [66]:
orders_date_check = orders.copy()

ORDER_DATE_COLUMNS = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

date_range_rows = []

for column_name in ORDER_DATE_COLUMNS:
    converted_date = pd.to_datetime(
        orders_date_check[column_name],
        errors="coerce"
    )

    non_null_before_conversion = orders_date_check[column_name].notna()

    conversion_failure_count = int(
        (non_null_before_conversion & converted_date.isna()).sum()
    )

    date_range_rows.append(
        {
            "column_name": column_name,
            "original_missing_count": int(
                orders_date_check[column_name].isna().sum()
            ),
            "conversion_failure_count": conversion_failure_count,
            "earliest_date": converted_date.min(),
            "latest_date": converted_date.max()
        }
    )

date_range_results = pd.DataFrame(date_range_rows)

display(date_range_results)

,column_name,original_missing_count,conversion_failure_count,earliest_date,latest_date
0,order_purchase_timestamp,0,0,2016-09-04 21:15:19,2018-10-17 17:30:18
1,order_approved_at,160,0,2016-09-15 12:16:38,2018-09-03 17:40:06
2,order_delivered_carrier_date,1783,0,2016-10-08 10:34:01,2018-09-11 19:48:28
3,order_delivered_customer_date,2965,0,2016-10-11 13:46:32,2018-10-17 13:22:46
4,order_estimated_delivery_date,0,0,2016-09-30 00:00:00,2018-11-12 00:00:00


In [67]:
order_status_distribution = (
    orders["order_status"]
    .value_counts(dropna=False)
    .rename_axis("order_status")
    .reset_index(name="order_count")
)

display(order_status_distribution)

,order_status,order_count
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [68]:
print("1. Product category summary")
product_category_summary = pd.DataFrame(
    {
        "total_product_rows": [len(products)],
        "non_missing_category_rows": [
            int(products["product_category_name"].notna().sum())
        ],
        "missing_category_rows": [
            int(products["product_category_name"].isna().sum())
        ],
        "unique_non_missing_categories": [
            int(products["product_category_name"].nunique())
        ]
    }
)

display(product_category_summary)

print("\n2. Payment type distribution")
payment_type_distribution = (
    order_payments["payment_type"]
    .value_counts(dropna=False)
    .rename_axis("payment_type")
    .reset_index(name="payment_record_count")
)

display(payment_type_distribution)

print("\n3. Review score distribution")
review_score_distribution = (
    order_reviews["review_score"]
    .value_counts(dropna=False)
    .rename_axis("review_score")
    .reset_index(name="review_record_count")
)

display(review_score_distribution)

1. Product category summary


,total_product_rows,non_missing_category_rows,missing_category_rows,unique_non_missing_categories
0,32951,32341,610,73



2. Payment type distribution


,payment_type,payment_record_count
0,credit_card,76795
1,boleto,19784
2,voucher,5775
3,debit_card,1529
4,not_defined,3



3. Review score distribution


,review_score,review_record_count
0,5,57328
1,4,19142
2,1,11424
3,3,8179
4,2,3151


In [69]:
def validate_relationship(
    child_dataframe,
    child_column,
    parent_dataframe,
    parent_column,
    relationship_name
):
    child_values = child_dataframe[child_column]
    parent_values = parent_dataframe[parent_column].dropna()

    child_non_missing_values = child_values.dropna()

    unmatched_values = child_non_missing_values[
        ~child_non_missing_values.isin(parent_values)
    ]

    total_child_rows = len(child_dataframe)
    child_null_count = int(child_values.isna().sum())
    child_non_null_count = int(child_non_missing_values.shape[0])
    unmatched_non_null_count = int(unmatched_values.shape[0])

    if child_non_null_count == 0:
        matched_percentage = None
    else:
        matched_percentage = round(
            (child_non_null_count - unmatched_non_null_count)
            / child_non_null_count
            * 100,
            2
        )

    if unmatched_non_null_count == 0:
        result_status = "PASS — all non-null child values found in parent table"
    else:
        result_status = "CHECK — unmatched child values found"

    return {
        "relationship": relationship_name,
        "child_rows": total_child_rows,
        "child_null_count": child_null_count,
        "child_non_null_count": child_non_null_count,
        "unmatched_non_null_count": unmatched_non_null_count,
        "matched_percentage": matched_percentage,
        "sample_unmatched_values": ", ".join(
            map(str, unmatched_values.head(5).tolist())
        ),
        "result_status": result_status
    }


RELATIONSHIPS_TO_CHECK = [
    {
        "relationship": "orders.customer_id → customers.customer_id",
        "child_dataframe": orders,
        "child_column": "customer_id",
        "parent_dataframe": customers,
        "parent_column": "customer_id"
    },
    {
        "relationship": "order_items.order_id → orders.order_id",
        "child_dataframe": order_items,
        "child_column": "order_id",
        "parent_dataframe": orders,
        "parent_column": "order_id"
    },
    {
        "relationship": "order_payments.order_id → orders.order_id",
        "child_dataframe": order_payments,
        "child_column": "order_id",
        "parent_dataframe": orders,
        "parent_column": "order_id"
    },
    {
        "relationship": "order_reviews.order_id → orders.order_id",
        "child_dataframe": order_reviews,
        "child_column": "order_id",
        "parent_dataframe": orders,
        "parent_column": "order_id"
    },
    {
        "relationship": "order_items.product_id → products.product_id",
        "child_dataframe": order_items,
        "child_column": "product_id",
        "parent_dataframe": products,
        "parent_column": "product_id"
    },
    {
        "relationship": "order_items.seller_id → sellers.seller_id",
        "child_dataframe": order_items,
        "child_column": "seller_id",
        "parent_dataframe": sellers,
        "parent_column": "seller_id"
    },
    {
        "relationship": (
            "products.product_category_name → "
            "category_translation.product_category_name"
        ),
        "child_dataframe": products,
        "child_column": "product_category_name",
        "parent_dataframe": category_translation,
        "parent_column": "product_category_name"
    }
]

relationship_check_rows = []

for relationship_check in RELATIONSHIPS_TO_CHECK:
    result = validate_relationship(
        child_dataframe=relationship_check["child_dataframe"],
        child_column=relationship_check["child_column"],
        parent_dataframe=relationship_check["parent_dataframe"],
        parent_column=relationship_check["parent_column"],
        relationship_name=relationship_check["relationship"]
    )

    relationship_check_rows.append(result)

relationship_check_results = pd.DataFrame(relationship_check_rows)

display(relationship_check_results)

,relationship,child_rows,child_null_count,child_non_null_count,unmatched_non_null_count,matched_percentage,sample_unmatched_values,result_status
0,orders.customer_id → customers.customer_id,99441,0,99441,0,100.00,,PASS — all non-null child values found in pare...
1,order_items.order_id → orders.order_id,112650,0,112650,0,100.00,,PASS — all non-null child values found in pare...
2,order_payments.order_id → orders.order_id,103886,0,103886,0,100.00,,PASS — all non-null child values found in pare...
3,order_reviews.order_id → orders.order_id,99224,0,99224,0,100.00,,PASS — all non-null child values found in pare...
4,order_items.product_id → products.product_id,112650,0,112650,0,100.00,,PASS — all non-null child values found in pare...
5,order_items.seller_id → sellers.seller_id,112650,0,112650,0,100.00,,PASS — all non-null child values found in pare...
6,products.product_category_name → category_tran...,32951,610,32341,13,99.96,"pc_gamer, portateis_cozinha_e_preparadores_de_...",CHECK — unmatched child values found


In [70]:
ZIP_CODE_COVERAGE_CHECKS = [
    {
        "relationship": (
            "customers.customer_zip_code_prefix → "
            "geolocation.geolocation_zip_code_prefix"
        ),
        "child_dataframe": customers,
        "child_column": "customer_zip_code_prefix",
        "parent_dataframe": geolocation,
        "parent_column": "geolocation_zip_code_prefix"
    },
    {
        "relationship": (
            "sellers.seller_zip_code_prefix → "
            "geolocation.geolocation_zip_code_prefix"
        ),
        "child_dataframe": sellers,
        "child_column": "seller_zip_code_prefix",
        "parent_dataframe": geolocation,
        "parent_column": "geolocation_zip_code_prefix"
    }
]

zip_code_coverage_rows = []

for zip_check in ZIP_CODE_COVERAGE_CHECKS:
    result = validate_relationship(
        child_dataframe=zip_check["child_dataframe"],
        child_column=zip_check["child_column"],
        parent_dataframe=zip_check["parent_dataframe"],
        parent_column=zip_check["parent_column"],
        relationship_name=zip_check["relationship"]
    )

    zip_code_coverage_rows.append(result)

zip_code_coverage_results = pd.DataFrame(zip_code_coverage_rows)

display(zip_code_coverage_results)

print(
    "Important: This is a coverage check only. "
    "Do not treat geolocation_zip_code_prefix as a strict unique primary key."
)

,relationship,child_rows,child_null_count,child_non_null_count,unmatched_non_null_count,matched_percentage,sample_unmatched_values,result_status
0,customers.customer_zip_code_prefix → geolocati...,99441,0,99441,278,99.72,"72300, 11547, 64605, 72465, 7729",CHECK — unmatched child values found
1,sellers.seller_zip_code_prefix → geolocation.g...,3095,0,3095,7,99.77,"82040, 91901, 72580, 2285, 7412",CHECK — unmatched child values found


Important: This is a coverage check only. Do not treat geolocation_zip_code_prefix as a strict unique primary key.


## Phase 1 Notebook Completion Note

This notebook only documents raw-data understanding and validation.

Completed checks:
- Raw CSV file existence and file names
- Table sizes and columns
- Data types and sample records
- Key-field missing values and duplicates
- Missing-value overview
- Date ranges and categorical values
- Core table relationship checks

Not completed in this notebook:
- Data cleaning
- Dropping or filling missing values
- SQL database import
- Business metrics
- Visualisation
- Business conclusions